**Author:** Salvador Navas  
**Date:** 2025-06-27

### PERSIANN-CCS

**PERSIANN-CCS** is a satellite-based precipitation estimation product from the **CHRS at UC Irvine**.

- **Spatial resolution:** ~0.04° (~4 km)
- **Temporal resolution:** 1 hour
- **Temporal coverage:** 2003–present
- **Spatial coverage:** 60°S–60°N

> This notebook uses **synthetic demo data** that reproduces the PERSIANN format
> without requiring FTP access to the UCI server.
> Replace the synthetic section with actual `PERSSIANDownloader()` calls when FTP access is available.

In [1]:
# PERSSIANDownloader uses FTP access to the UCI server (no credentials needed)
# Interactive map requires 'ipyleaflet' — skipped
# from pyhydra.data_sources.rainfall import PERSSIANDownloader
# from pyhydra.utils import interactive_map

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr
import tempfile, os
print('Imports OK')

Imports OK


---
## PERSSIANDownloader — real usage

```python
from pyhydra.data_sources.rainfall import PERSSIANDownloader

# Point mode
dl = PERSSIANDownloader(lon=-0.62, lat=39.39, path_output='persiann_point')
series = dl.download_hourly('2024-10-29', '2024-10-30')

# Area mode
dl = PERSSIANDownloader(lon_min=-3.25, lon_max=1.41, lat_min=37.2, lat_max=40.45,
                        path_output='persiann_area')
dl.download_daily('2024-10-01', '2024-10-31')
```

In [2]:
# === SYNTHETIC DEMO DATA ===
# Creates an xarray Dataset matching PERSIANN-CCS format
# Variables: prcp (mm/hr)  Coords: lat, lon, time

TMPDIR = tempfile.mkdtemp()
rng = np.random.default_rng(42)

# Hourly data for October 2024 (24 h × 7 days)
times = pd.date_range('2024-10-29 00:00', periods=48, freq='1h')
lats  = np.arange(40.44, 37.19, -0.04)   # PERSIANN 0.04° grid
lons  = np.arange(-3.26, 1.41,  0.04)

# Convective precipitation event pattern
lon_grid, lat_grid = np.meshgrid(lons, lats)
event_center_lat, event_center_lon = 39.0, -0.5
dist = np.sqrt((lat_grid - event_center_lat)**2 + (lon_grid - event_center_lon)**2)
spatial_pattern = np.exp(-dist**2 / 2)

prcp = np.zeros((len(times), len(lats), len(lons)), dtype='float32')
for i, t in enumerate(times):
    hour = t.hour
    if 10 <= hour <= 20:   # afternoon convection
        intensity = rng.uniform(0, 15) * np.sin(np.pi * (hour - 10) / 10)
        prcp[i] = (intensity * spatial_pattern + rng.uniform(0, 0.5, spatial_pattern.shape)).astype('float32')

ds_persiann = xr.Dataset(
    {'prcp': (['time', 'lat', 'lon'], prcp)},
    coords={
        'time': times,
        'lat':  lats,
        'lon':  lons,
    }
)
ds_persiann['prcp'].attrs.update({
    'long_name': 'PERSIANN-CCS hourly precipitation',
    'units': 'mm/hr',
    'missing_value': -9999,
})

NC_FILE = os.path.join(TMPDIR, 'persiann_demo.nc')
ds_persiann.to_netcdf(NC_FILE)
print(f'Synthetic PERSIANN NetCDF created: {NC_FILE}')
print(ds_persiann)

Synthetic PERSIANN NetCDF created: /var/folders/44/dw7p6q9108xcc4mmh_f7q1vc0000gn/T/tmpvc836h1o/persiann_demo.nc
<xarray.Dataset> Size: 2MB
Dimensions:  (time: 48, lat: 82, lon: 117)
Coordinates:
  * time     (time) datetime64[ns] 384B 2024-10-29 ... 2024-10-30T23:00:00
  * lat      (lat) float64 656B 40.44 40.4 40.36 40.32 ... 37.28 37.24 37.2
  * lon      (lon) float64 936B -3.26 -3.22 -3.18 -3.14 ... 1.26 1.3 1.34 1.38
Data variables:
    prcp     (time, lat, lon) float32 2MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0


---
## 1. Extract point time series

In [3]:
# Extract precipitation at Valencia (39.39°N, -0.62°E)
LAT, LON = 39.39, -0.62

ds = xr.open_dataset(NC_FILE)
ts = ds['prcp'].sel(lat=LAT, lon=LON, method='nearest')
df_pt = ts.to_dataframe().reset_index()[['time', 'prcp']]
df_pt = df_pt.rename(columns={'time': 'datetime', 'prcp': 'precip_mm_hr'})
df_pt = df_pt.set_index('datetime').sort_index()
print(df_pt.describe())
df_pt.head()

       precip_mm_hr
count     48.000000
mean       1.942833
std        3.490964
min        0.000000
25%        0.000000
50%        0.000000
75%        2.660139
max       13.103211


,precip_mm_hr
datetime,
2024-10-29 00:00:00,0.0
2024-10-29 01:00:00,0.0
2024-10-29 02:00:00,0.0
2024-10-29 03:00:00,0.0
2024-10-29 04:00:00,0.0


---
## 2. Spatial mean and totals

In [4]:
# Hourly mean over the domain
area_mean = ds['prcp'].mean(dim=['lat', 'lon'])
df_area = area_mean.to_dataframe().rename(columns={'prcp': 'domain_mean_mm_hr'})
daily_accum = df_area.resample('D').sum() * 1   # already mm/hr, sum gives mm/day if each step = 1 hr
print('\nDaily accumulated precipitation (mm):')
print(daily_accum.round(2))


Daily accumulated precipitation (mm):
            domain_mean_mm_hr
time                         
2024-10-29          21.969999
2024-10-30          17.070000


---
## 3. Visualisation

In [5]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(df_pt.index, df_pt['precip_mm_hr'], lw=1.0, color='navy')
axes[0].set_ylabel('Precipitation (mm/hr)')
axes[0].set_title(
    f'PERSIANN-CCS — Valencia ({LAT}°N, {LON}°E) — synthetic demo', fontsize=13
)
axes[0].grid(alpha=0.3)

# Spatial plot of total accumulated precipitation over all hours
total_precip = ds['prcp'].sum(dim='time').values
img = axes[1].imshow(
    total_precip, origin='upper', aspect='auto',
    extent=[lons.min(), lons.max(), lats.min(), lats.max()],
    cmap='Blues'
)
plt.colorbar(img, ax=axes[1], label='Total precipitation (mm)')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_title('Accumulated precipitation — Oct 29-30 2024')
axes[1].plot(LON, LAT, 'r*', markersize=12, label='Valencia')
axes[1].legend()

plt.tight_layout()
plt.savefig('/tmp/PERSSIAN_download.png', dpi=100, bbox_inches='tight')
plt.close()
print('Plot saved to /tmp/PERSSIAN_download.png')
ds.close()

Plot saved to /tmp/PERSSIAN_download.png
